In [1]:
import random
import torch
from torch import nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils

!pip install opencv-python==4.9.0.80

Either use the commands below, or create a new conda environment with

```
conda env create -f environment_supermario.yml
conda activate mario-rl
```

In [25]:
!pip install numpy==1.26.4 gym==0.26.2 gym-super-mario-bros==7.4.0 nes-py==8.2.1 torch torchvision opencv-python pyvirtualdisplay matplotlib

## If you run on colab and you want to visualize graphics, do it like so: if not, do not run this cell

In [ ]:
runOnColab=False

if runOnColab:
    ! apt install freeglut3 freeglut3-dev

import matplotlib.pyplot as plt
from IPython import display as ipythondisplay

from pyvirtualdisplay import Display
display = Display(visible=0, size=(400, 300))
display.start()

## Now we are ready

In [2]:
import numpy as np
import torch
import gym
import gym_super_mario_bros
from gym_super_mario_bros.actions import SIMPLE_MOVEMENT
from nes_py.wrappers import JoypadSpace
from gym.wrappers import GrayScaleObservation, ResizeObservation

print("Gym:", gym.__version__)


Gym: 0.26.2


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [3]:
def make_mario_env(world: int = 1, stage: int = 1):
    """
    Create a Super Mario Bros environment with:
      - Discrete SIMPLE_MOVEMENT action space.
      - Grayscale 84x84 observations.
    """
    env_id = f"SuperMarioBros-{world}-{stage}-v3"
    env = gym_super_mario_bros.make(env_id, 
        apply_api_compatibility=True,
        render_mode="human",  # optional but nice for seeing the game
    )
    env = JoypadSpace(env, SIMPLE_MOVEMENT)
    env = GrayScaleObservation(env, keep_dim=False)  # (H, W)
    env = ResizeObservation(env, 84)                 # (84, 84)
    return env

In [4]:
def preprocess_state(state, device):
    """
    Flatten and normalize (84,84) grayscale image to (1, 84*84) tensor.
    """
    state = np.array(state, dtype=np.float32) / 255.0
    state = state.flatten()
    return torch.tensor(state, dtype=torch.float32, device=device).unsqueeze(0) #torch.from_numpy(state).unsqueeze(0).to(device)

In [5]:
class DQN(nn.Module):
    """Simple feed-forward DQN with flattened image input."""
    def __init__(self, input_dim: int, n_actions: int):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 256)
        self.fc3 = nn.Linear(256, n_actions)
    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [6]:
import random
from collections import deque

In [7]:
class ReplayMemory:
    """Fixed-size replay buffer for experience replay."""
    def __init__(self, capacity: int):
        self.memory = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))
    def sample(self, batch_size: int):
        return random.sample(self.memory, batch_size)
    def __len__(self):
        return len(self.memory)

In [8]:
class MarioDQNAgent:
    """DQN-based agent for Super Mario Bros (solution version, classic Gym API)."""
    def __init__(
        self,
        n_actions: int,
        input_dim: int = 84 * 84,
        lr: float = 1e-4,
        gamma: float = 0.99,
        replay_capacity: int = 50000,
        device: str = "cpu",
    ):
        self.n_actions = n_actions
        self.device = device
        self.gamma = gamma
        self.policy_net = DQN(input_dim, n_actions).to(device)
        self.target_net = DQN(input_dim, n_actions).to(device)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()
        self.optimizer = torch.optim.Adam(self.policy_net.parameters(), lr=lr)
        self.memory = ReplayMemory(replay_capacity)
    def select_action(self, state, epsilon: float) -> int:
        """Epsilon-greedy action selection."""
        if random.random() < epsilon:
            return random.randrange(self.n_actions)
        with torch.no_grad():
            q_values = self.policy_net(state)
            return int(q_values.argmax(dim=1).item())
    def optimize_model(self, batch_size: int):
        """One DQN update step using replay memory."""
        if len(self.memory) < batch_size:
            return
        transitions = self.memory.sample(batch_size)
        states, actions, rewards, next_states, dones = zip(*transitions)
        state_batch = torch.cat(states).to(self.device)
        next_state_batch = torch.cat(next_states).to(self.device)
        action_batch = torch.tensor(actions, device=self.device).long().unsqueeze(1)
        reward_batch = torch.tensor(rewards, device=self.device).float().unsqueeze(1)
        done_batch = torch.tensor(dones, device=self.device).float().unsqueeze(1)
        q_values = self.policy_net(state_batch)
        state_action_values = q_values.gather(1, action_batch)
        with torch.no_grad():
            next_q_values = self.target_net(next_state_batch)
            max_next_q, _ = next_q_values.max(dim=1, keepdim=True)
            targets = reward_batch + self.gamma * max_next_q * (1.0 - done_batch)
        loss = F.smooth_l1_loss(state_action_values, targets)
        self.optimizer.zero_grad()
        loss.backward()
        nn_utils.clip_grad_norm_(self.policy_net.parameters(), 10.0)
        self.optimizer.step()

In [9]:
def train_dqn(
    num_episodes: int = 200,
    batch_size: int = 32,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.1,
    epsilon_decay_episodes: int = 150,
    target_update_interval: int = 1000,
) -> MarioDQNAgent:
    """Train a DQN agent on Super Mario Bros with classic Gym API."""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Using device: {device}")
    env = make_mario_env()
    n_actions = env.action_space.n
    agent = MarioDQNAgent(n_actions=n_actions, device=device)
    global_step = 0
    for episode in range(num_episodes):
        obs, info = env.reset()
        state = preprocess_state(obs, device)
        done = False
        total_reward = 0.0
        frac = min(episode / float(epsilon_decay_episodes), 1.0)
        epsilon = epsilon_start + frac * (epsilon_end - epsilon_start)
        while not done:
            action = agent.select_action(state, epsilon)
            next_obs, reward, terminated, truncated, info = env.step(action)
            done = terminated or truncated
            next_state = preprocess_state(next_obs, device)
            agent.memory.push(state, action, reward, next_state, done)
            state = next_state
            total_reward += reward
            global_step += 1
            agent.optimize_model(batch_size)
            if global_step % target_update_interval == 0:
                agent.target_net.load_state_dict(agent.policy_net.state_dict())
        print(f"Episode {episode+1}/{num_episodes}  Reward: {total_reward:.1f}  Epsilon: {epsilon:.3f}")
    env.close()
    return agent

In [10]:
def run_agent_episode(agent: MarioDQNAgent, world: int = 1, stage: int = 1, render: bool = True) -> float:
    env = make_mario_env(world, stage)
    obs, info = env.reset()
    state = preprocess_state(obs, agent.device)
    done = False
    total_reward = 0.0
    while not done:
        if render:
            env.render()
        action = agent.select_action(state, epsilon=0.0)
        next_obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        next_state = preprocess_state(next_obs, agent.device)
        total_reward += reward
        state = next_state
    env.close()
    print(f"[Agent] Total reward: {total_reward:.2f}")
    return total_reward

In [11]:
def run_random_episode(world: int = 1, stage: int = 1, render: bool = True) -> float:
    env = make_mario_env(world, stage)
    obs, info = env.reset()
    done = False
    total_reward = 0.0
    while not done:
        if render:
            env.render()
        action = env.action_space.sample()
        obs, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        total_reward += reward
    env.close()
    print(f"[Random] Total reward: {total_reward:.2f}")
    return total_reward

In [12]:
def evaluate_agent_vs_random(agent: MarioDQNAgent, episodes: int = 5):
    agent_scores = []
    random_scores = []
    for i in range(episodes):
        print(f"\n=== Match {i+1}/{episodes} ===")
        agent_scores.append(run_agent_episode(agent, render=False))
        random_scores.append(run_random_episode(render=False))
    print("\n=== Summary (computer vs computer) ===")
    print(f"Agent average reward:  {np.mean(agent_scores):.2f}")
    print(f"Random average reward: {np.mean(random_scores):.2f}")

In [13]:
def run_human_episode(world: int = 1, stage: int = 1):
    env = make_mario_env(world, stage)
    obs, info = env.reset()
    done = False
    total_reward = 0.0
    print("\nActions (SIMPLE_MOVEMENT):")
    for idx, action in enumerate(SIMPLE_MOVEMENT):
        print(f"{idx}: {action}")
    print("Press Ctrl+C to quit.")
    while not done:
        env.render()
        try:
            a = int(input("Choose action index: "))
            if a < 0 or a >= len(SIMPLE_MOVEMENT):
                print("Invalid index, try again.")
                continue
        except ValueError:
            print("Please enter a valid integer.")
            continue
        obs, reward, terminated, truncated, info = env.step(a)
        done = terminated or truncated
        total_reward += reward
    env.close()
    print(f"[Human] Total reward: {total_reward:.2f}")

In [ ]:
agent = train_dqn(num_episodes=20)
run_agent_episode(agent, render=True)
evaluate_agent_vs_random(agent, episodes=3)

Using device: cpu


/opt/miniconda3/envs/supermario/lib/python3.10/site-packages/gym/envs/registration.py:627: UserWarning: WARN: The environment creator metadata doesn't include `render_modes`, contains: ['render.modes', 'video.frames_per_second']
  logger.warn(
/opt/miniconda3/envs/supermario/lib/python3.10/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


Episode 1/20  Reward: 501.0  Epsilon: 1.000
